# 18 — SOLID Principles

## Objectives
- Understand and apply all 5 SOLID principles
- Recognize violations and how to fix them
- Apply principles to real Java code

## SOLID
| Letter | Principle | Key Rule |
|--------|-----------|----------|
| S | Single Responsibility | One class, one job |
| O | Open/Closed | Open for extension, closed for modification |
| L | Liskov Substitution | Subclasses honor parent contracts |
| I | Interface Segregation | Small, specific interfaces |
| D | Dependency Inversion | Depend on abstractions |

In [1]:
// O — Open/Closed Principle + D — Dependency Inversion
interface DiscountStrategy { double apply(double price); }

class NoDiscount       implements DiscountStrategy { public double apply(double p) { return p; } }
class StudentDiscount  implements DiscountStrategy { public double apply(double p) { return p * 0.80; } }
class SeniorDiscount   implements DiscountStrategy { public double apply(double p) { return p * 0.70; } }
class SeasonalDiscount implements DiscountStrategy { public double apply(double p) { return p * 0.85; } }

// PriceCalculator never changes when new discount type is added
class PriceCalculator {
    private final DiscountStrategy strategy; // DIP: depends on abstraction
    PriceCalculator(DiscountStrategy strategy) { this.strategy = strategy; }
    double calculate(double basePrice) { return strategy.apply(basePrice); }
}

// I — Interface Segregation
interface Readable  { String read(); }
interface Writable  { void write(String data); }
interface Deletable { void delete(); }

// FileReader only needs Readable
class TextFileReader implements Readable {
    public String read() { return "File content..."; }
}

// ReadWriteFile needs both
class TextFile implements Readable, Writable {
    private String content = "";
    public String read() { return content; }
    public void write(String data) { content += data; System.out.println("Written: " + data); }
}

// Demo
double basePrice = 10000.0;
System.out.println("Base price: INR " + basePrice);

DiscountStrategy[] strategies = {new NoDiscount(), new StudentDiscount(), 
                                  new SeniorDiscount(), new SeasonalDiscount()};
String[] names = {"No Discount", "Student", "Senior", "Seasonal"};

for (int i = 0; i < strategies.length; i++) {
    PriceCalculator calc = new PriceCalculator(strategies[i]);
    System.out.printf("%-15s: INR %.2f%n", names[i], calc.calculate(basePrice));
}

System.out.println();
TextFile file = new TextFile();
file.write("Hello SOLID!");
System.out.println("Read: " + file.read());

Base price: INR 10000.0
No Discount    : INR 10000.00
Student        : INR 8000.00
Senior         : INR 7000.00
Seasonal       : INR 8500.00

Written: Hello SOLID!
Read: Hello SOLID!


## Mini Challenge
Refactor a `NotificationService` that handles Email, SMS, and Push in one class to follow SRP, OCP, and DIP.

In [2]:
import java.util.*;

// ============================================================================
// D — Dependency Inversion Principle (Abstraction)
// ============================================================================
interface NotificationChannel {
    void send(String recipient, String message);
}

// ============================================================================
// S — Single Responsibility & O — Open/Closed Principle (Implementations)
// ============================================================================
class EmailNotification implements NotificationChannel {
    @Override
    public void send(String recipient, String message) {
        // Only responsible for Email protocol details
        System.out.println("📧 Sending Email to " + recipient + ": " + message);
    }
}

class SmsNotification implements NotificationChannel {
    @Override
    public void send(String recipient, String message) {
        // Only responsible for SMS protocol/gateway details
        System.out.println("💬 Sending SMS to " + recipient + ": " + message);
    }
}

class PushNotification implements NotificationChannel {
    @Override
    public void send(String recipient, String message) {
        // Only responsible for Push notification/FCM details
        System.out.println("🔔 Sending Push Notification to " + recipient + ": " + message);
    }
}

// ============================================================================
// High-Level Module (DIP compliant: depends on Abstraction, not Concretions)
// ============================================================================
class NotificationService {
    // Aggregates channels dynamically. We can add/remove channels without modifying this class.
    private final List<NotificationChannel> channels;

    // Injecting channels via Constructor (DIP)
    public NotificationService(List<NotificationChannel> channels) {
        this.channels = channels;
    }

    public void broadcast(String user, String message) {
        for (NotificationChannel channel : channels) {
            channel.send(user, message);
        }
    }
}

// ============================================================================
// Demo Execution
// ============================================================================

// Build our list of channels (easily customizable/injectable via Spring or manual setup)
List<NotificationChannel> activeChannels = Arrays.asList(
    new EmailNotification(),
    new SmsNotification(),
    new PushNotification()
);

// Instantiate the service with the channels injected
NotificationService service = new NotificationService(activeChannels);

System.out.println("--- Broadcasting Security Alert ---");
service.broadcast("user_dev_01", "Unusual login activity detected on your Mac.");

--- Broadcasting Security Alert ---
📧 Sending Email to user_dev_01: Unusual login activity detected on your Mac.
💬 Sending SMS to user_dev_01: Unusual login activity detected on your Mac.
🔔 Sending Push Notification to user_dev_01: Unusual login activity detected on your Mac.
